# Pràctica progressiva — Anàlisi de vendes d'una botiga

## Encàrrec professional
Treballes com a tècnic/a d'anàlisi de dades d'una botiga. La persona responsable comercial t'ha lliurat un CSV de vendes i necessita un resum fiable per prendre decisions. Hauràs de preparar les dades, justificar les regles aplicades i comunicar els resultats a un públic no tècnic.

## Objectius
- Llegir CSV amb `csv.DictReader` i explorar-ne l'estructura.
- Validar, normalitzar i convertir dades amb funcions pròpies.
- Separar dades vàlides, corregibles i rebutjades amb traçabilitat.
- Calcular indicadors i fer agrupacions.
- Verificar els resultats i generar fitxers d'informe.
- Formular conclusions prudents i defensar les decisions.

**Fitxers:** posa `vendes_botiga.csv` al mateix directori que aquest notebook.

La pràctica es divideix en fases guiades. Utilitza només la biblioteca estàndard de Python.

## Fase 1 — Inspecció guiada

Comença per entendre el fitxer abans de modificar-lo. Cada fila és una línia de venda; els camps són textuals quan es llegeixen del CSV.

In [10]:
import csv
from datetime import datetime

nom_fitxer = "vendes_botiga.csv"

# EXERCI 1.0: Obre el fitxer 'nom_fitxer' en mode lectura, llegeix-lo amb csv.DictReader
# i desa els camps a la variable 'camps' i els registres com una llista a 'registres'.

with open(nom_fitxer, mode="r", encoding="utf-8", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

# Comprovacions
print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
if registres:
    print("Exemple de primer registre:", registres[0])

Camps del CSV: ['data      ', ' producte  ', ' categoria  ', ' unitats', ' preu_unitari', ' ciutat']
Nombre total de registres: 19
Exemple de primer registre: {'data      ': '2026-01-03', ' producte  ': ' Llibreta  ', ' categoria  ': ' Papereria  ', ' unitats': ' 12     ', ' preu_unitari': ' 2.50        ', ' ciutat': ' Barcelona'}


### Exercici 1.1
Calcula quants registres tenen almenys un camp buit o format només per espais. No modifiquis les dades encara.

**Pista:** Pots fer servir `.strip() == ""` sobre cada valor del diccionari per comprovar si un camp està buit o té només espais.

In [43]:
# GUIA PAS A PAS:
# 1. Inicialitza el comptador 'registres_incomplets' a 0.
# 2. Revisa cada registre de la llista 'registres'.
# 3. Comprova si algun dels seus valors està buit després d'aplicar .strip().
# 4. Incrementa el comptador si es detecta algun camp buit.

# registres_incomplets = 0

# for reg in registres:
#     # TODO: Comprova si hi ha algun camp buit en la fila reg
#     # Pista: utilitza any(...) o un bucle sobre reg.values()
#     pass


# print(f"Registres amb almenys un camp buit/espais: {registres_incomplets}")


registres_incomplets = 0

for reg in registres:
    if any(
        str(valor).strip() == ""
        for valor in reg.values()
    ):
        registres_incomplets += 1

print(f"Registres amb almenys un camp buit/espais: {registres_incomplets}")





Registres amb almenys un camp buit/espais: 2


### Exercici 1.2

**Reflexiona:** quines dades són imprescindibles per calcular l'import d'una venda? Quines són necessàries per segmentar-la/classificar-la?

## Fase 2 — Regles de qualitat i neteja guiada

Cal aplicar als registres de vendes aquestes regles:
1. Eliminar espais sobrants dels camps textuals (`strip()`).
2. `unitats` ha de ser un enter estrictament positiu (> 0).
3. `preu_unitari` ha de ser un nombre positiu (> 0); s'accepten coma o punt decimal.
4. La data ha de tenir el format `AAAA-MM-DD` i ser una data vàlida.
5. Producte, categoria i ciutat són camps descriptius obligatoris.
6. Un registre amb valors numèrics o data no interpretables no entra als càlculs. No inventis valors per completar-los (imputació).

### Exercici 2 — Funcions de conversió amb gestió d'errors
Implementa funcions de conversió robustes utilitzant blocs `try...except`.

In [62]:
# def converteix_enter_positiu(valor: str):
#     """Retorna un int positiu (>0) o None si no és vàlid."""
#     try:
#         v = int(str(valor).strip())
#         return v if v > 0 else None
#     except (ValueError, TypeError):
#          return None

# def converteix_preu(valor: str):
#     """Accepta coma o punt decimal; retorna float positiu (>0) o None si no és vàlid."""
#     # TODO: Neteja espais, reemplaça ',' per '.', converteix a float i comprova que sigui > 0
#     pass

# def data_valida(valor: str) -> bool:
#     """Retorna True si el text correspon a una data vàlida en format AAAA-MM-DD."""
#     # TODO: Utilitza datetime.strptime(..., "%Y-%m-%d") dins d'un try-except
#     pass

from datetime import datetime

def converteix_enter_positiu(valor: str):
    """Retorna un int positiu (>0) o None si no és vàlid."""
    try:
        v = int(str(valor).strip())
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None


def converteix_preu(valor: str):
    """Accepta coma o punt decimal; retorna float positiu (>0) o None si no és vàlid."""
    try:
        text = str(valor).strip().replace(",", ".")
        v = float(text)
        return v if v > 0 and v != float("inf") and v != float("-inf") and v == v else None
    except (ValueError, TypeError):
        return None


def data_valida(valor: str) -> bool:
    """Retorna True si el text correspon a una data vàlida en format AAAA-MM-DD."""
    try:
        text = str(valor).strip()
        data = datetime.strptime(text, "%Y-%m-%d")
        return data.strftime("%Y-%m-%d") == text
    except (ValueError, TypeError):
        return False


### Exercici 3 — Neteja amb traçabilitat

Classifica les dades en tres estructures:
- `vendes_netes`: llista de diccionaris amb dades convertides i camp `import` inclòs.
- `rebutjades`: llista de registres no vàlids.
- `incidencies`: llista de registres/incidències amb el detall per a la traçabilitat (`fila_csv`, `tipus`, `camp`, `valor_original`, `accio`).

In [ ]:

from datetime import datetime
import math

# FUNCIONS DE VALIDACIO I CONVERSIO

def converteix_enter_positiu(valor):
    try:
        v = int(str(valor).strip())
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None


def converteix_preu(valor):
    try:
        text = str(valor).strip().replace(",", ".")
        v = float(text)
        return v if math.isfinite(v) and v > 0 else None
    except (ValueError, TypeError):
        return None


def data_valida(valor):
    try:
        text = str(valor).strip()
        data = datetime.strptime(text, "%Y-%m-%d")
        return data.strftime("%Y-%m-%d") == text
    except (ValueError, TypeError):
        return False


# PAS 0: NETEJA DE LES DADES I DE LES COLUMNES


registres_nets = []

for reg in registres:
    # Ignorar registres que no siguin diccionaris
    if not isinstance(reg, dict):
        continue

    reg_nou = {}

    for clau, valor in reg.items():
        # Ignorar claus buides o None
        if clau is None:
            continue

        clau_neta = str(clau).strip()

        if clau_neta != "":
            reg_nou[clau_neta] = valor

    registres_nets.append(reg_nou)

registres = registres_nets


# --------------------------------------------------
# PAS 1-5: VALIDACIO I CLASSIFICACIO
# --------------------------------------------------

vendes_netes = []
incidencies = []
rebutjades = []

obligatoris = ["producte", "categoria", "ciutat"]

for idx, reg in enumerate(registres, start=2):
    es_valid = True
    reg_net = {}

    # PAS 1: Camps obligatoris
    for camp in obligatoris:
        val_original = reg.get(camp, "")
        val_net = str(val_original or "").strip()

        if not val_net:
            es_valid = False
            incidencies.append({
                "fila_csv": idx,
                "tipus": "REBUTJAT",
                "camp": camp,
                "valor_original": val_original,
                "accio": "Camp obligatori buit"
            })
        else:
            if str(val_original) != val_net:
                incidencies.append({
                    "fila_csv": idx,
                    "tipus": "CORRECCIO",
                    "camp": camp,
                    "valor_original": val_original,
                    "accio": "Espais sobrants eliminats"
                })

            reg_net[camp] = val_net

    # PAS 2: Unitats
    val_unitats = reg.get("unitats", "")
    unitats = converteix_enter_positiu(val_unitats)

    if unitats is None:
        es_valid = False
        incidencies.append({
            "fila_csv": idx,
            "tipus": "REBUTJAT",
            "camp": "unitats",
            "valor_original": val_unitats,
            "accio": "Unitats no valides: cal un enter positiu"
        })
    else:
        reg_net["unitats"] = unitats

        if str(val_unitats) != str(val_unitats).strip():
            incidencies.append({
                "fila_csv": idx,
                "tipus": "CORRECCIO",
                "camp": "unitats",
                "valor_original": val_unitats,
                "accio": "Espais sobrants eliminats"
            })

    # PAS 3: Preu unitari
    val_preu = reg.get("preu_unitari", "")
    preu = converteix_preu(val_preu)

    if preu is None:
        es_valid = False
        incidencies.append({
            "fila_csv": idx,
            "tipus": "REBUTJAT",
            "camp": "preu_unitari",
            "valor_original": val_preu,
            "accio": "Preu no valid: cal un nombre positiu"
        })
    else:
        reg_net["preu_unitari"] = preu

        if "," in str(val_preu):
            incidencies.append({
                "fila_csv": idx,
                "tipus": "CORRECCIO",
                "camp": "preu_unitari",
                "valor_original": val_preu,
                "accio": "Separador decimal convertit de coma a punt"
            })
        elif str(val_preu) != str(val_preu).strip():
            incidencies.append({
                "fila_csv": idx,
                "tipus": "CORRECCIO",
                "camp": "preu_unitari",
                "valor_original": val_preu,
                "accio": "Espais sobrants eliminats"
            })

    # PAS 4: Data
    val_data = reg.get("data", "")

    if not data_valida(val_data):
        es_valid = False
        incidencies.append({
            "fila_csv": idx,
            "tipus": "REBUTJAT",
            "camp": "data",
            "valor_original": val_data,
            "accio": "Data invalida o format incorrecte"
        })
    else:
        reg_net["data"] = str(val_data).strip()

    # PAS 5: Classificacio final
    if es_valid:
        reg_net["import"] = round(
            reg_net["unitats"] * reg_net["preu_unitari"], 2
        )
        vendes_netes.append(reg_net)
    else:
        rebutjades.append(reg)


# --------------------------------------------------
# RESULTATS
# --------------------------------------------------

print(f"Llegits: {len(registres)}")
print(f"Valids: {len(vendes_netes)}")
print(f"Rebutjats: {len(rebutjades)}")
print(f"Incidencies documentades: {len(incidencies)}")

print("\nColumnes detectades:")
if registres:
    print(list(registres[0].keys()))

print("\nPrimer registre net:")
if registres:
    print(registres[0])

print("\nPrimeres incidencies:")
for incidencia in incidencies[:10]:
    print(incidencia)

print("\nPrimeres vendes valides:")
for venda in vendes_netes[:5]:
    print(venda)


Llegits: 19
Valids: 15
Rebutjats: 4
Incidencies documentades: 95

Columnes detectades:
['data', 'producte', 'categoria', 'unitats', 'preu_unitari', 'ciutat']

Primer registre net:
{'data': '2026-01-03', 'producte': ' Llibreta  ', 'categoria': ' Papereria  ', 'unitats': ' 12     ', 'preu_unitari': ' 2.50        ', 'ciutat': ' Barcelona'}

Primeres incidencies:
{'fila_csv': 2, 'tipus': 'CORRECCIO', 'camp': 'producte', 'valor_original': ' Llibreta  ', 'accio': 'Espais sobrants eliminats'}
{'fila_csv': 2, 'tipus': 'CORRECCIO', 'camp': 'categoria', 'valor_original': ' Papereria  ', 'accio': 'Espais sobrants eliminats'}
{'fila_csv': 2, 'tipus': 'CORRECCIO', 'camp': 'ciutat', 'valor_original': ' Barcelona', 'accio': 'Espais sobrants eliminats'}
{'fila_csv': 2, 'tipus': 'CORRECCIO', 'camp': 'unitats', 'valor_original': ' 12     ', 'accio': 'Espais sobrants eliminats'}
{'fila_csv': 2, 'tipus': 'CORRECCIO', 'camp': 'preu_unitari', 'valor_original': ' 2.50        ', 'accio': 'Espais sobrants elim

## Fase 3 — Indicadors de negoci

Calcula els indicadors globals i realitza les agrupacions per categoria i ciutat.

In [64]:
# GUIA DE CÀLCUL:
# 1. Suma total d'unitats i d'imports a partir de 'vendes_netes'.
# 2. Càlcul de mitjanes: preu unitari mitjà vs import mitjà per línia.
# 3. Agrupació en diccionaris acumulant la facturació per clau.

# 1. Calcula els indicadors totals
total_unitats = sum(v["unitats"] for v in vendes_netes)

facturacio_total = round(
    sum(v["import"] for v in vendes_netes), 2
)

# 2. Calcula les mitjanes
if vendes_netes:
    preu_mitja = round(
        sum(v["preu_unitari"] for v in vendes_netes) / len(vendes_netes), 2
    )

    import_mitja = round(
        facturacio_total / len(vendes_netes), 2
    )
else:
    preu_mitja = 0
    import_mitja = 0

# 3. Agrupa la facturacio per categoria i ciutat
resum_categoria = {}
resum_ciutat = {}

for v in vendes_netes:
    cat = v["categoria"]
    ciu = v["ciutat"]

    resum_categoria[cat] = round(
        resum_categoria.get(cat, 0.0) + v["import"], 2
    )

    resum_ciutat[ciu] = round(
        resum_ciutat.get(ciu, 0.0) + v["import"], 2
    )

# 4. Mostra els resultats
print(f"Total unitats: {total_unitats}")
print(f"Facturació total: {facturacio_total} €")
print(f"Preu unitari mitjà: {preu_mitja} €")
print(f"Import mitjà per línia: {import_mitja} €")
print("Facturació per categoria:", resum_categoria)
print("Facturació per ciutat:", resum_ciutat)


Total unitats: 128
Facturació total: 894.6 €
Preu unitari mitjà: 15.55 €
Import mitjà per línia: 59.64 €
Facturació per categoria: {'Papereria': 150.5, 'Accessoris': 245.4, 'Electrònica': 498.7}
Facturació per ciutat: {'Barcelona': 479.3, 'Girona': 189.0, 'Tarragona': 201.3, 'Lleida': 25.0}


### Control de qualitat
Validació automàtica de la consistència de les dades mitjançant assercions (`assert`). Omple la informació corresponent per verificar les teves dades.

In [65]:
# 
# TODO 1: Comprova que la suma de vàlids i rebutjats
# sigui igual al total llegit
assert len(vendes_netes) + len(rebutjades) == len(registres), \
    "Error: La suma de vàlids i rebutjats no quadra amb el total."


# TODO 2: Comprova que unitats, preu_unitari i import
# siguin tots positius (> 0)
assert all(
    v["unitats"] > 0
    and v["preu_unitari"] > 0
    and v["import"] > 0
    for v in vendes_netes
), "Error: Hi ha valors nuls o negatius."


# TODO 3: Comprova que la suma de les categories
# coincideixi amb la facturació total
assert abs(
    sum(resum_categoria.values()) - facturacio_total
) < 0.01, "Error en la suma de categories."


print("Totes les comprovacions han estat superades!")


Totes les comprovacions han estat superades!


## Fase 4 — Interpretació i comunicació

Respon a les preguntes analítiques identificant els màxims i avaluant les conclusions de negoci.

In [66]:
# 
# TODO 1: Troba la categoria i ciutat amb més facturació

top_categoria = (
    max(resum_categoria.items(), key=lambda x: x[1])
    if resum_categoria else (None, 0)
)

top_ciutat = (
    max(resum_ciutat.items(), key=lambda x: x[1])
    if resum_ciutat else (None, 0)
)


# TODO 2: Troba el producte amb més unitats venudes

unitats_per_producte = {}

for v in vendes_netes:
    producte = v["producte"]
    unitats = v["unitats"]

    unitats_per_producte[producte] = (
        unitats_per_producte.get(producte, 0) + unitats
    )

top_producte = (
    max(unitats_per_producte.items(), key=lambda x: x[1])
    if unitats_per_producte else (None, 0)
)


# Mostra els resultats
print(f"1. Categoria amb més facturació: {top_categoria[0]} ({top_categoria[1]} €)")
print(f"   Ciutat amb més facturació: {top_ciutat[0]} ({top_ciutat[1]} €)")
print(f"2. Producte amb més unitats venudes: {top_producte[0]} ({top_producte[1]} unitats)")


1. Categoria amb més facturació: Electrònica (498.7 €)
   Ciutat amb més facturació: Barcelona (479.3 €)
2. Producte amb més unitats venudes: Bolígraf (65 unitats)


## Fase 5 — Lliurament professional

Generació dels tres fitxers de sortida: `informe_vendes.txt`, `incidencies.csv` i `vendes_netes.csv`.

In [67]:

import csv
from datetime import datetime

nom_informe = "informe_vendes.txt"
nom_incidencies = "incidencies.csv"
nom_netes = "vendes_netes.csv"


# TODO 1: Genera vendes_netes.csv
if vendes_netes:
    with open(nom_netes, mode="w", encoding="utf-8-sig", newline="") as f:
        escriptor = csv.DictWriter(
            f,
            fieldnames=vendes_netes[0].keys(),
            delimiter=";"
        )
        escriptor.writeheader()
        escriptor.writerows(vendes_netes)
else:
    with open(nom_netes, mode="w", encoding="utf-8-sig", newline="") as f:
        escriptor = csv.DictWriter(
            f,
            fieldnames=[
                "data", "producte", "categoria", "unitats",
                "preu_unitari", "ciutat", "import"
            ],
            delimiter=";"
        )
        escriptor.writeheader()


# TODO 2: Genera incidencies.csv
camps_incidencies = [
    "fila_csv",
    "tipus",
    "camp",
    "valor_original",
    "accio"
]

with open(nom_incidencies, mode="w", encoding="utf-8-sig", newline="") as f:
    escriptor = csv.DictWriter(
        f,
        fieldnames=camps_incidencies,
        delimiter=";"
    )
    escriptor.writeheader()
    escriptor.writerows(incidencies)


# TODO 3: Genera informe_vendes.txt
with open(nom_informe, mode="w", encoding="utf-8", newline="") as f:
    f.write("INFORME EXECUTIU DE VENDES\n")
    f.write("=" * 35 + "\n\n")

    # Data de generació
    data_generacio = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    f.write(f"Data de generacio: {data_generacio}\n\n")

    # Cobertura de dades
    total_llegits = len(registres)
    total_valids = len(vendes_netes)
    total_rebutjats = len(rebutjades)

    percent_valids = (
        total_valids / total_llegits * 100
        if total_llegits else 0
    )
    percent_rebutjats = (
        total_rebutjats / total_llegits * 100
        if total_llegits else 0
    )

    f.write("1. COBERTURA DE DADES\n")
    f.write("-" * 25 + "\n")
    f.write(f"Registres llegits: {total_llegits}\n")
    f.write(f"Registres valids: {total_valids} ({percent_valids:.2f}%)\n")
    f.write(
        f"Registres rebutjats: {total_rebutjats} "
        f"({percent_rebutjats:.2f}%)\n"
    )
    f.write(f"Incidencies documentades: {len(incidencies)}\n\n")

    # Indicadors globals
    f.write("2. INDICADORS GLOBALS\n")
    f.write("-" * 25 + "\n")
    f.write(f"Total d'unitats venudes: {total_unitats}\n")
    f.write(f"Facturacio total: {facturacio_total:.2f} EUR\n")
    f.write(f"Preu unitari mitja: {preu_mitja:.2f} EUR\n")
    f.write(f"Import mitja per linia: {import_mitja:.2f} EUR\n\n")

    # Desglossament per categoria
    f.write("3. FACTURACIO PER CATEGORIA\n")
    f.write("-" * 30 + "\n")
    for categoria, import_total in sorted(
        resum_categoria.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        f.write(f"{categoria}: {import_total:.2f} EUR\n")

    f.write("\n4. FACTURACIO PER CIUTAT\n")
    f.write("-" * 27 + "\n")
    for ciutat, import_total in sorted(
        resum_ciutat.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        f.write(f"{ciutat}: {import_total:.2f} EUR\n")

    # Millors resultats
    f.write("\n5. RESULTATS DESTACATS\n")
    f.write("-" * 25 + "\n")

    if top_categoria[0] is not None:
        f.write(
            f"Categoria amb mes facturacio: {top_categoria[0]} "
            f"({top_categoria[1]:.2f} EUR)\n"
        )

    if top_ciutat[0] is not None:
        f.write(
            f"Ciutat amb mes facturacio: {top_ciutat[0]} "
            f"({top_ciutat[1]:.2f} EUR)\n"
        )

    if top_producte[0] is not None:
        f.write(
            f"Producte amb mes unitats venudes: {top_producte[0]} "
            f"({top_producte[1]} unitats)\n"
        )


print("Fitxers d'informe i exportació generats correctament.")
print(f"- {nom_netes}")
print(f"- {nom_incidencies}")
print(f"- {nom_informe}")


Fitxers d'informe i exportació generats correctament.
- vendes_netes.csv
- incidencies.csv
- informe_vendes.txt
